# Step 14 -- CARE-Sim Structured World Model

Trains an ensemble of GPT-style transformer world models on ICU patient trajectories using the structured variant.
Architecture: CareSimGPT (adapted from Trajectory Transformer, Janner et al. NeurIPS 2021).

## Before running

**Step 1:** From the repo root, run this once locally to create the upload zip:
```
python scripts/prepare_colab_upload.py
```
This creates `caresim_colab.zip` in the repo root.

**Step 2:** Upload two files to Google Drive (drag and drop into `MyDrive/CareAI/`):
- `caresim_colab.zip`
- `data/processed/icu_readmit/rl_dataset_tier2.parquet` -> into `MyDrive/CareAI/data/`

**Step 3:** Runtime -> Change runtime type -> **T4 GPU**. Then run all cells.

In [ ]:
# Cell 1: Mount Drive + check GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU -- training will be slow. Consider Runtime > Change runtime type > T4 GPU.')

In [ ]:
# Cell 2: Unzip source files + verify data
import os, sys, zipfile

DRIVE_ROOT = '/content/drive/MyDrive/CareAI'
ZIP_PATH   = os.path.join(DRIVE_ROOT, 'caresim_colab.zip')
UNZIP_DIR  = '/content/caresim_src'          # unzipped here, not on Drive (faster I/O)
SRC_PATH   = os.path.join(UNZIP_DIR, 'src')
DATA_PATH  = os.path.join(DRIVE_ROOT, 'data', 'rl_dataset_tier2.parquet')
MODEL_DIR  = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', 'caresim_structured')

SMOKE_SCRIPT = os.path.join(UNZIP_DIR, 'scripts', 'icu_readmit', 'step_14_caresim_smoke_test.py')
TRAIN_SCRIPT = os.path.join(UNZIP_DIR, 'scripts', 'icu_readmit', 'step_14_caresim_train_structured.py')

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f'caresim_colab.zip not found at {ZIP_PATH}. Run prepare_colab_upload.py locally first.')

print(f'Unzipping {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(UNZIP_DIR)
print(f'Unzipped to {UNZIP_DIR}')

for pkg_dir in [
    os.path.join(SRC_PATH, 'careai'),
    os.path.join(SRC_PATH, 'careai', 'icu_readmit'),
    os.path.join(SRC_PATH, 'careai', 'icu_readmit', 'caresim'),
]:
    init = os.path.join(pkg_dir, '__init__.py')
    if not os.path.exists(init):
        open(init, 'w').close()

os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Data file not found:\n  {DATA_PATH}\n'
        'Upload rl_dataset_tier2.parquet to MyDrive/CareAI/data/'
    )

size_gb = os.path.getsize(DATA_PATH) / 1e9
print(f'Data file OK ({size_gb:.2f} GB)')
print(f'Model output: {MODEL_DIR}')
print('Ready.')

In [ ]:
# Cell 3: Smoke test (synthetic data, ~30 seconds)
import subprocess, sys

def run_streaming(cmd, env):
    """Run a command and stream output line by line in real-time."""
    proc = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

print('Running smoke test...')
rc = run_streaming(
    [sys.executable, '-u', SMOKE_SCRIPT],
    env={**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'},
)
if rc != 0:
    raise RuntimeError('Smoke test FAILED -- do not proceed to full training.')
print('Smoke test PASSED.')

In [ ]:
# Cell 4: Full training
# Trains 5 ensemble members x 30 epochs. Output streams live as it runs.
# GPU estimated time: ~30-40 min. CPU: ~8h.

BASE_ARGS = [
    sys.executable, '-u', TRAIN_SCRIPT,
    '--data',        DATA_PATH,
    '--save-dir',    MODEL_DIR,
    '--device',      device,
    '--state-dim',   '8',
    '--action-dim',  '4',
    '--n-models',    '5',
    '--d-model',     '256',
    '--n-heads',     '8',
    '--n-layers',    '4',
    '--n-epochs',    '30',
    '--batch-size',  '64',
    '--lr',          '1e-3',
]

ENV = {**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'}

print('=' * 60)
print('FULL TRAINING -- structured CARE-Sim (8-state Tier-2)')
print('=' * 60)
rc = run_streaming(BASE_ARGS, env=ENV)
if rc != 0:
    raise RuntimeError('Structured training FAILED.')
print('\nStructured training complete.')



In [ ]:
# Cell 5: Verify outputs
print('Files saved to Drive:')
for root, dirs, files in os.walk(MODEL_DIR):
    for f in sorted(files):
        path = os.path.join(root, f)
        size_kb = os.path.getsize(path) / 1024
        rel = os.path.relpath(path, MODEL_DIR)
        print(f'  {rel:55s}  {size_kb:8.1f} KB')

print('\nDownload models/icu_readmit/caresim_structured/ from Drive to:')
print('  CareAI/models/icu_readmit/caresim_structured/')

